# Phase 2 · Read-only GQA/MHA BIO CRF ensemble

Compare original GQA BIO best (epoch 6) with original MHA BIO best (epoch 1), without retraining or modifying either checkpoint. Both must match ordered BIO tags, tokenizer, deterministic 512/384 BPE windows, word alignment/ownership, and CRF constraints.

Validation is fixed to the preserved six documents and 781 gold entities. The fixed GQA weights are 1.00, 0.00, 0.50, and 0.75. Interpolate complete CRF sequence scores (emissions, transitions, start and end) before Viterbi. Cache each model's validation emissions and CRF scores separately on Drive; collect models sequentially and atomically resume per window. Cache schema v2 stores every word's logits as `[words,tags]` and uses a new cache identity, leaving any older cache untouched.

Test export is separate. Default selection is the validation winner only if it beats GQA-only; a user may explicitly request one fixed weight later, with its validation score shown. Test results never select the mixture. No training or Codabench submission occurs.


In [ ]:
# 1 · Fixed experiment settings; no training action
from pathlib import Path
ACTION = "Evaluate"  # Inspect | Evaluate | Export test
GQA_CHECKPOINT_CHOICE = "best_epoch6"  # best_epoch6 | latest_epoch15 (explicit optional comparison)
EXPORT_GQA_WEIGHT = None  # None = validation winner; or explicitly 1.0, 0.75, 0.5, or 0.0
COURSE_ROOT = Path("/content/drive/MyDrive/legal-llm-course")
TRAIN_DIR = COURSE_ROOT / "Phase_2_LNER/Train"
TEST_TEXT = COURSE_ROOT / "Phase_2_LNER/Test/fold3-text.json"
GQA_BEST = COURSE_ROOT / "runs/phase2_lner/bio_bilstm_crf/gqa_pre/best.pt"
GQA_LATEST15 = COURSE_ROOT / "runs/phase2_lner/bio_bilstm_crf/gqa_pre_latest_continuation_5ep/latest.pt"
MHA_BEST = COURSE_ROOT / "runs/phase2_lner/bio_bilstm_crf/mha_pre/best.pt"
if GQA_CHECKPOINT_CHOICE not in ("best_epoch6", "latest_epoch15"):
    raise ValueError('GQA_CHECKPOINT_CHOICE must be best_epoch6 or latest_epoch15')
GQA_CHECKPOINT = GQA_BEST if GQA_CHECKPOINT_CHOICE == "best_epoch6" else GQA_LATEST15
MHA_CHECKPOINT = MHA_BEST
ENSEMBLE_ROOT = COURSE_ROOT / "runs/phase2_lner/bio_bilstm_crf/gqa_mha_bio_ensemble"
RUN_ROOT = ENSEMBLE_ROOT / GQA_CHECKPOINT_CHOICE
BIO_HELPER = COURSE_ROOT / "bio_crf.py"
ENSEMBLE_HELPER = COURSE_ROOT / "bio_ensemble.py"
MAX_BPE, STRIDE_BPE = 512, 384
SEED, VALIDATION_FRACTION = 17, .15
LSTM_HIDDEN = 256
GQA_WEIGHT_GRID = (1.0, 0.0, 0.5, 0.75)
EXPECTED_VALIDATION_DOCS, EXPECTED_VALIDATION_GOLD = 6, 781


In [ ]:
# 2 · Mount Drive and load existing data/model helpers
from google.colab import drive
drive.mount('/content/drive')
import sys, shutil, hashlib, json, importlib.util, subprocess, zipfile, random, math, time, gc
import torch
BUNDLE_ZIP = COURSE_ROOT / "phase2_lner_gqa_repaired_bundle.zip"
LOCAL = Path('/content/phase2-bio-crf-ensemble')
if not BUNDLE_ZIP.is_file(): raise FileNotFoundError(f'Existing repaired bundle missing: {BUNDLE_ZIP}')
if not BIO_HELPER.is_file(): raise FileNotFoundError(f'BIO helper missing: {BIO_HELPER}')
if not ENSEMBLE_HELPER.is_file(): raise FileNotFoundError(f'Ensemble helper missing: {ENSEMBLE_HELPER}')
LOCAL.mkdir(parents=True,exist_ok=True)
with zipfile.ZipFile(BUNDLE_ZIP) as archive: archive.extractall(LOCAL)
shutil.copy2(BIO_HELPER,LOCAL/'bio_crf.py'); shutil.copy2(ENSEMBLE_HELPER,LOCAL/'bio_ensemble.py')
sys.path.insert(0,str(LOCAL))
missing=[]
if importlib.util.find_spec('tokenizers') is None: missing.append('tokenizers==0.23.1')
if importlib.util.find_spec('huggingface_hub') is None: missing.append('huggingface_hub')
if missing: subprocess.check_call([sys.executable,'-m','pip','install','-q',*missing])
from tokenizers import Tokenizer
from legal_ner import *
from bio_crf import *
from bio_ensemble import *
TOKENIZER_PATH=LOCAL/'tokenizer/tokenizer.json'
TOKENIZER_SHA256=hashlib.sha256(TOKENIZER_PATH.read_bytes()).hexdigest()
if TOKENIZER_SHA256!='906194848c1632431aac7d186e7c4f49da83cb6cbd480fa222c8472401cbf8ab':
    raise ValueError('tokenizer identity mismatch')
TOKENIZER=Tokenizer.from_file(str(TOKENIZER_PATH))
print('Existing tokenizer, data/BIO helpers, and ensemble helper loaded.')


In [ ]:
# 3 · Download the pinned dataset files if needed, then audit exact pairs
from google.colab import userdata
training_files=ensure_training_files(COURSE_ROOT,userdata.get('HF_TOKEN'))
print('Training files ready:',[str(p.relative_to(COURSE_ROOT)) for p in training_files])
files=sorted(p for p in TRAIN_DIR.rglob('*') if p.is_file())
print('Files:',[(p.name,p.stat().st_size) for p in files])
labels_path=next((p for p in files if p.name.lower()=='labels.txt'),None)
if labels_path is None: raise FileNotFoundError('actual labels.txt not found')
load_labels(labels_path)
text_files={p.name[:-10]:p for p in files if p.name.endswith('-text.json')}
gold_files={p.name[:-10]:p for p in files if p.name.endswith('-gold.json')}
if not text_files or set(text_files)!=set(gold_files):
    raise ValueError(f'Expected paired official *-text.json/*-gold.json folds; text={sorted(text_files)}, gold={sorted(gold_files)}')
rows=[]
for fold in sorted(text_files):
    try: rows.extend(load_fold_pair(text_files[fold],gold_files[fold]))
    except Exception as e: raise ValueError(f'Could not join actual {fold} fold: {e}') from e
if len({r['id'] for r in rows})!=len(rows): raise ValueError('document ID appears in more than one fold')
report=audit(rows)
print(json.dumps(report,ensure_ascii=False,indent=2))
assert tuple(report['label_counts'])==LABELS
if report['substring_evidence_entities']:
    print('End-exclusive Python Unicode offsets verified against every provided entity_text value.')
else:
    print('WARNING: no entity_text evidence exists; offset convention is not independently proven by the annotations.')


In [ ]:
# 4 · Preserved 64/6 document split and deterministic windows
train_rows,val_rows=split_documents(rows,VALIDATION_FRACTION,SEED)
if (len(train_rows),len(val_rows))!=(64,6): raise ValueError(f'expected 64/6 split; got {len(train_rows)}/{len(val_rows)}')
assert {r['id'] for r in train_rows}.isdisjoint({r['id'] for r in val_rows})
train_windows=[w for row in train_rows for w in make_document_windows(row,TOKENIZER,MAX_BPE,STRIDE_BPE)]
val_windows=[w for row in val_rows for w in make_document_windows(row,TOKENIZER,MAX_BPE,STRIDE_BPE)]
gold_count=sum(len(r['entities']) for r in val_rows)
if len(val_rows)!=EXPECTED_VALIDATION_DOCS or gold_count!=EXPECTED_VALIDATION_GOLD:
    raise ValueError(f'fixed validation must be 6 docs/781 gold; got {len(val_rows)}/{gold_count}')
print({'train_documents':64,'validation_documents':6,'train_windows':len(train_windows),
       'validation_windows':len(val_windows),'validation_gold_entities':gold_count})


## Immutable checkpoint provenance

Primary pair: `runs/phase2_lner/bio_bilstm_crf/gqa_pre/best.pt` (epoch 6) and `runs/phase2_lner/bio_bilstm_crf/mha_pre/best.pt` (epoch 1). Optional GQA latest-epoch-15 is a separate explicit choice, off by default. Checkpoints are only read; caches/results/export live in `gqa_mha_bio_ensemble/`.


In [ ]:
# 5 · Configure exact model architectures and checkpoint selection
from legal_scaffold.models import ModelConfig,CausalDecoder
MODEL_CONFIGS={
 'gqa_pre':{'vocab_size':32768,'d_model':512,'layers':12,'heads':8,'kv_heads':2,'ffn_hidden':2048,'context_length':2048,'attention':'gqa','norm_placement':'pre','tie_embeddings':True,'rope_base':10000,'norm_epsilon':1e-6,'init_std':.02},
 'mha_pre':{'vocab_size':32768,'d_model':512,'layers':12,'heads':8,'kv_heads':8,'ffn_hidden':2048,'context_length':2048,'attention':'mha','norm_placement':'pre','tie_embeddings':True,'rope_base':10000,'norm_epsilon':1e-6,'init_std':.02}}
CHECKPOINTS={'gqa':{'preset':'gqa_pre','path':GQA_CHECKPOINT,'expected_epoch':6 if GQA_CHECKPOINT_CHOICE=='best_epoch6' else 15},
             'mha':{'preset':'mha_pre','path':MHA_CHECKPOINT,'expected_epoch':1}}
if ACTION!='Inspect':
    if not torch.cuda.is_available(): raise RuntimeError('Evaluate requires CUDA; no CPU fallback or training')
    for path in (GQA_CHECKPOINT,MHA_CHECKPOINT):
        if not path.is_file(): raise FileNotFoundError(path)
print('GQA selection:',GQA_CHECKPOINT_CHOICE,GQA_CHECKPOINT)
print('MHA source:',MHA_CHECKPOINT)
print('No optimizer/training loop is created.')


In [ ]:
# 6 · Atomic resumable caches, sequential model inference, and fixed mixtures
def window_digest(windows):
    h=hashlib.sha256()
    for w in windows:
        h.update(str(w['document_id']).encode()); h.update(str((w['word_start_index'],w['word_end_index'])).encode())
        h.update(json.dumps(w['input_ids'],separators=(',',':')).encode())
        h.update(json.dumps((w['word_last'],w['word_spans'],w['owner_start_index'],w['owner_end_index']),separators=(',',':')).encode())
    return h.hexdigest()

def cache_one(name,split_name,windows):
    meta=CHECKPOINTS[name]; path=meta['path']; st=path.stat()
    payload=torch.load(path,map_location='cpu',weights_only=False)
    if payload.get('schema_version')!=1: raise ValueError(f'{name} checkpoint schema mismatch')
    identity=payload.get('identity',{}); progress=payload.get('progress',{})
    if identity.get('preset')!=meta['preset'] or identity.get('tokenizer_sha256')!=TOKENIZER_SHA256:
        raise ValueError(f'{name} checkpoint/tokenizer identity mismatch')
    if tuple(identity.get('labels',()))!=tuple(LABELS): raise ValueError(f'{name} ordered label list mismatch')
    epoch=int(progress.get('best_epoch',progress.get('epoch',-1)))
    if name=='gqa' and GQA_CHECKPOINT_CHOICE=='latest_epoch15':
        epoch=int(progress.get('parent_latest_epoch',-1))+int(progress.get('epoch',-1))
        if int(progress.get('successful_updates',-1))!=6405: raise ValueError('GQA epoch-15 update count mismatch')
    if epoch!=meta['expected_epoch']: raise ValueError(f'{name} expected epoch {meta["expected_epoch"]}, found {epoch}')
    print(f'{name} checkpoint epoch {epoch}; progress:',progress,flush=True)
    cache_identity={'cache_schema_version':2,'name':name,'path':str(path),'size':st.st_size,'mtime_ns':st.st_mtime_ns,
      'checkpoint_identity':identity,'progress':progress,'tokenizer_sha256':TOKENIZER_SHA256,
      'split':split_name,'window_count':len(windows),'window_digest':window_digest(windows),
      'max_bpe':MAX_BPE,'stride_bpe':STRIDE_BPE}
    cache_path=RUN_ROOT/'cache'/f'{split_name}_{name}_{cache_key(cache_identity)}.pt'
    config=MODEL_CONFIGS[meta['preset']]
    model=BackboneBiLSTMCRF(CausalDecoder(ModelConfig(**config)),config['d_model'],LABELS,LSTM_HIDDEN)
    model.load_state_dict(payload['model'],strict=True); del payload
    model.to('cuda').eval()
    cached=collect_emission_cache(model,windows,cache_path,cache_identity,'cuda',name+' '+split_name)
    del model; gc.collect(); torch.cuda.empty_cache()
    return cached,{'path':str(path),'identity':identity,'progress':progress,'cache':str(cache_path)}

TAGS=bio_tags(LABELS)
if ACTION in ('Evaluate','Export test'):
    gqa_val,gqa_provenance=cache_one('gqa','validation',val_windows)
    mha_val,mha_provenance=cache_one('mha','validation',val_windows)
    validation_results={}
    for weight in GQA_WEIGHT_GRID:
        result=evaluate_weight(val_windows,val_rows,gqa_val,mha_val,weight,LABELS,TAGS,decode_entities)
        if weight in (0.0,1.0) and not result['endpoint_decoder_parity']:
            raise AssertionError(f'endpoint decoder parity failed for GQA weight {weight}')
        validation_results[weight]=result
        print(f'GQA/MHA={weight:.2f}/{1-weight:.2f}; macro-F1={result["macro_f1"]:.12f}; parity={result["endpoint_decoder_parity"]}; cached CUDA-path differences={result["cached_reference_path_device_mismatches"]}')
else: print('Inspect only: model loading and inference skipped.')


In [ ]:
# 7 · Save fixed validation comparison; no training
if ACTION in ('Evaluate','Export test'):
    validation_winner_weight=max(GQA_WEIGHT_GRID,key=lambda w:validation_results[w]['macro_f1'])
    baseline_f1=validation_results[1.0]['macro_f1']; winner_f1=validation_results[validation_winner_weight]['macro_f1']
    winner_beats_gqa=winner_f1>baseline_f1
    compact={str(w):{k:v for k,v in r.items() if k!='predictions'} for w,r in validation_results.items()}
    report={'validation_documents':6,'validation_gold_entities':781,
      'document_ids':[str(r['id']) for r in val_rows],'checkpoint_provenance':{'gqa':gqa_provenance,'mha':mha_provenance},
      'gqa_weights':list(GQA_WEIGHT_GRID),'results':compact,'gqa_baseline_macro_f1':baseline_f1,
      'validation_winner_gqa_weight':validation_winner_weight,'validation_winner_macro_f1':winner_f1,
      'validation_winner_beats_gqa':winner_beats_gqa,'test_used_for_selection':False}
    RUN_ROOT.mkdir(parents=True,exist_ok=True)
    result_path=RUN_ROOT/'validation_results.json'; temp=result_path.with_suffix('.json.tmp')
    temp.write_text(json.dumps(report,ensure_ascii=False,indent=2)+'\n',encoding='utf-8'); temp.replace(result_path)
    print('Validation winner:',validation_winner_weight,'macro-F1:',winner_f1,'GQA baseline:',baseline_f1,'improved:',winner_beats_gqa)
    print('Full per-label/per-document results:',result_path)
    for w in GQA_WEIGHT_GRID:
        print('weight',w,'per-label',validation_results[w]['per_label'])
        print('weight',w,'per-document',validation_results[w]['per_document'])
    print('No checkpoint or optimizer was changed; no training performed.')
else: print('Evaluation skipped.')


In [ ]:
# 8 · Validation summary
if ACTION in ('Evaluate','Export test'):
    print('Same validation set:',len(val_rows),'documents /',EXPECTED_VALIDATION_GOLD,'gold')
    print('GQA-only macro-F1:',validation_results[1.0]['macro_f1'])
    print('Validation-selected GQA weight:',validation_winner_weight,'beats baseline:',winner_beats_gqa)


In [10]:
# 9 · Separate official test export; test never selects a weight
if ACTION=='Export test':
    if EXPORT_GQA_WEIGHT is None:
        if not winner_beats_gqa:
            raise ValueError('Validation winner does not beat GQA-only. Set EXPORT_GQA_WEIGHT explicitly to one fixed weight to request it anyway.')
        selected_weight=float(validation_winner_weight); reason='validation winner, higher than GQA baseline'
    else:
        selected_weight=float(EXPORT_GQA_WEIGHT)
        if selected_weight not in GQA_WEIGHT_GRID: raise ValueError(f'weight must be in {GQA_WEIGHT_GRID}')
        reason='explicit user choice; validation score shown, test not used for selection'
    print('Requested export mix:',selected_weight,1-selected_weight,reason)
    print('Validation score:',validation_results[selected_weight]['macro_f1'],'GQA baseline:',validation_results[1.0]['macro_f1'])
    if not TEST_TEXT.is_file(): raise FileNotFoundError(TEST_TEXT)
    test_rows=load_records(TEST_TEXT)
    if len(test_rows)!=35: raise ValueError(f'expected all 35 test IDs, found {len(test_rows)}')
    test_ids=[str(r['id']) for r in test_rows]
    if len(set(test_ids))!=35 or '1305957' not in test_ids: raise ValueError('official test IDs mismatch')
    empty_ids=[str(r['id']) for r in test_rows if not r['text']]
    if '1305957' not in empty_ids: raise ValueError('1305957 must be retained as empty text')
    test_windows=[w for r in test_rows if r['text'] for w in make_document_windows(r,TOKENIZER,MAX_BPE,STRIDE_BPE)]
    gqa_test,gqa_test_prov=cache_one('gqa','test',test_windows)
    mha_test,mha_test_prov=cache_one('mha','test',test_windows)
    components=blend_components(gqa_test['components'],mha_test['components'],selected_weight)
    decoded=[]
    for i,w in enumerate(test_windows):
        logits=selected_weight*gqa_test['emissions'][i]+(1-selected_weight)*mha_test['emissions'][i]
        decoded.extend(decode_entities(w,viterbi_decode(logits,components),TAGS))
    unique=sorted(set(decoded),key=lambda x:(x[0],x[1],x[2],x[3]))
    slug=f"submission_gqa{int(selected_weight*100):03d}_mha{int((1-selected_weight)*100):03d}_{GQA_CHECKPOINT_CHOICE}"
    outdir=RUN_ROOT/slug
    if outdir.exists(): raise FileExistsError(f'refusing to overwrite existing output {outdir}')
    texts={r['id']:r['text'] for r in test_rows}
    pred_path,zip_path=export_prediction([r['id'] for r in test_rows],[(*x,1.0) for x in unique],texts,outdir)
    written=json.loads(pred_path.read_text(encoding='utf-8'))
    if list(written)!=test_ids or written.get('1305957')!=[]: raise ValueError('export schema/ID/empty-document check failed')
    with zipfile.ZipFile(zip_path) as archive:
        if archive.namelist()!=['prediction.json']: raise ValueError('ZIP must contain only root prediction.json')
    labelled=outdir/f'predictions_{slug}.zip'; zip_path.rename(labelled)
    manifest={'gqa_weight':selected_weight,'mha_weight':1-selected_weight,'selection_reason':reason,
      'validation_score':validation_results[selected_weight]['macro_f1'],'gqa_baseline':validation_results[1.0]['macro_f1'],
      'test_used_for_selection':False,'test_documents':len(test_rows),'empty_ids':empty_ids,
      'predicted_entities':len(unique),'gqa_checkpoint':gqa_test_prov,'mha_checkpoint':mha_test_prov,
      'prediction_json':str(pred_path),'prediction_zip':str(labelled)}
    (outdir/'ensemble_manifest.json').write_text(json.dumps(manifest,ensure_ascii=False,indent=2)+'\n',encoding='utf-8')
    print('Local export ready; nothing submitted externally.')
    print('JSON:',pred_path,'ZIP:',labelled,'entities:',len(unique),'empty IDs:',empty_ids)
else: print('Test export skipped. Select ACTION = "Export test" only after reviewing validation.')


Requested export mix: 0.5 0.5 explicit user choice; validation score shown, test not used for selection
Validation score: 0.4474461779651089 GQA baseline: 0.44647299767734355
gqa checkpoint epoch 6; progress: {'epoch': 6, 'next_position': 0, 'successful_updates': 7686, 'overflow_retries': 11, 'best_val_macro_f1': 0.44647299767734355, 'best_epoch': 6, 'epochs_without_improvement': 0, 'last_val_macro_f1': 0.44647299767734355}
gqa test cached 10/741 windows
gqa test cached 20/741 windows
gqa test cached 30/741 windows
gqa test cached 40/741 windows
gqa test cached 50/741 windows
gqa test cached 60/741 windows
gqa test cached 70/741 windows
gqa test cached 80/741 windows
gqa test cached 90/741 windows
gqa test cached 100/741 windows
gqa test cached 110/741 windows
gqa test cached 120/741 windows
gqa test cached 130/741 windows
gqa test cached 140/741 windows
gqa test cached 150/741 windows
gqa test cached 160/741 windows
gqa test cached 170/741 windows
gqa test cached 180/741 windows
gqa t

In [8]:
# One-cell recovery for the already-open Colab runtime after the endpoint-parity error.
# It preserves the old (truncated) cache and writes fresh caches under cachefix_v2.
# Because the old cache kept only each window's first-word logits, the missing
# word logits cannot be reconstructed; this reloads both checkpoints sequentially
# and performs inference only (no training, no runtime allocation).
import importlib.util, shutil

assert 'cache_one' in globals() and 'val_windows' in globals() and 'val_rows' in globals(), \
    'Run this in the existing ensemble notebook runtime after the parity error.'
shutil.copy2(ENSEMBLE_HELPER, LOCAL / 'bio_ensemble.py')
spec = importlib.util.spec_from_file_location('bio_ensemble_recovered', LOCAL / 'bio_ensemble.py')
ensemble_fixed = importlib.util.module_from_spec(spec)
spec.loader.exec_module(ensemble_fixed)
# cache_one resolves these global names when called; explicitly rebind them so
# the stale `from bio_ensemble import *` definitions cannot be reused.
for name in ('cache_key', 'collect_emission_cache', 'blend_components',
             'evaluate_weight', 'viterbi_decode'):
    globals()[name] = getattr(ensemble_fixed, name)

OLD_RUN_ROOT = RUN_ROOT
RUN_ROOT = OLD_RUN_ROOT / 'cachefix_v2'
gqa_val, gqa_provenance = cache_one('gqa', 'validation', val_windows)
mha_val, mha_provenance = cache_one('mha', 'validation', val_windows)
validation_results = {}
for weight in GQA_WEIGHT_GRID:
    result = evaluate_weight(val_windows, val_rows, gqa_val, mha_val, weight,
                             LABELS, TAGS, decode_entities)
    if weight in (0.0, 1.0) and not result['endpoint_decoder_parity']:
        raise AssertionError(f'endpoint decoder parity failed after recache at GQA weight {weight}')
    validation_results[weight] = result
    print(f'GQA/MHA={weight:.2f}/{1-weight:.2f}; macro-F1={result["macro_f1"]:.12f}; '
          f'parity={result["endpoint_decoder_parity"]}; '
          f'cached CUDA-path differences={result["cached_reference_path_device_mismatches"]}')

validation_winner_weight = max(GQA_WEIGHT_GRID, key=lambda w: validation_results[w]['macro_f1'])
baseline_f1 = validation_results[1.0]['macro_f1']
winner_f1 = validation_results[validation_winner_weight]['macro_f1']
compact = {str(w): {k: v for k, v in r.items() if k != 'predictions'}
           for w, r in validation_results.items()}
report = {
    'validation_documents': 6, 'validation_gold_entities': 781,
    'document_ids': [str(r['id']) for r in val_rows],
    'checkpoint_provenance': {'gqa': gqa_provenance, 'mha': mha_provenance},
    'gqa_weights': list(GQA_WEIGHT_GRID), 'results': compact,
    'gqa_baseline_macro_f1': baseline_f1,
    'validation_winner_gqa_weight': validation_winner_weight,
    'validation_winner_macro_f1': winner_f1,
    'validation_winner_beats_gqa': winner_f1 > baseline_f1,
    'test_used_for_selection': False,
    'cache_recovery': 'schema v2; full [words,tags] emissions; old cache preserved',
}
RUN_ROOT.mkdir(parents=True, exist_ok=True)
result_path = RUN_ROOT / 'validation_results.json'
tmp_path = result_path.with_suffix('.json.tmp')
tmp_path.write_text(json.dumps(report, ensure_ascii=False, indent=2) + '\n', encoding='utf-8')
tmp_path.replace(result_path)
print('Fresh validation comparison saved:', result_path)


gqa checkpoint epoch 6; progress: {'epoch': 6, 'next_position': 0, 'successful_updates': 7686, 'overflow_retries': 11, 'best_val_macro_f1': 0.44647299767734355, 'best_epoch': 6, 'epochs_without_improvement': 0, 'last_val_macro_f1': 0.44647299767734355}
gqa validation cached 10/111 windows
gqa validation cached 20/111 windows
gqa validation cached 30/111 windows
gqa validation cached 40/111 windows
gqa validation cached 50/111 windows
gqa validation cached 60/111 windows
gqa validation cached 70/111 windows
gqa validation cached 80/111 windows
gqa validation cached 90/111 windows
gqa validation cached 100/111 windows
gqa validation cached 110/111 windows
gqa validation cached 111/111 windows
mha checkpoint epoch 1; progress: {'epoch': 1, 'next_position': 0, 'successful_updates': 1281, 'overflow_retries': 9, 'best_val_macro_f1': 0.4101461759510681, 'best_epoch': 1, 'epochs_without_improvement': 0, 'last_val_macro_f1': 0.4101461759510681}
mha validation cached 10/111 windows
mha validatio

In [9]:
ACTION = "Export test"
EXPORT_GQA_WEIGHT = 0.5

## Run steps

1. Select a T4 runtime and run all cells with `ACTION = "Evaluate"`. This performs no training, loads GQA/MHA sequentially, and caches validation emissions atomically on Drive. Schema v2 uses a fresh cache key and preserves older cache files.
2. Review the four fixed mixes, endpoint decoder parity, macro-F1, per-label/per-document results, and checkpoint provenance in `validation_results.json`.
3. For an official test comparison, set `ACTION = "Export test"`. By default this exports the validation winner only when it beats GQA-only. To request another fixed weight explicitly, set `EXPORT_GQA_WEIGHT` to `1.0`, `0.75`, `0.5`, or `0.0`; the manifest records the choice and validation score. Test results never select a weight.
4. The test exporter preserves all 35 IDs including empty `1305957`, uses the existing bounds/schema writer, and saves a separate labelled ZIP under `gqa_mha_bio_ensemble/`. It does not submit to Codabench.

The planned primary GQA source is epoch-6 best and MHA is epoch-1 best. The separate GQA epoch-15 latest checkpoint is available only by explicitly setting `GQA_CHECKPOINT_CHOICE = "latest_epoch15"`.


In [12]:
# Tiny CPU-only pilot for the already recovered six-document validation caches.
# Paste/run this as one new cell in the recovered ensemble Colab runtime.
# It does not load models, run GPU work, change ACTION/RUN_ROOT, write caches,
# or touch the in-progress 50/50 test export.
import torch

_REQUIRED = ('gqa_val', 'mha_val', 'val_windows', 'val_rows', 'LABELS', 'TAGS',
             'targets_and_segments', 'decode_entities', 'evaluate_weight')
_missing = [name for name in _REQUIRED if name not in globals()]
if _missing:
    raise RuntimeError(f'Open the recovered ensemble runtime first; missing: {_missing}')
if 'gqa_provenance' in globals() and 'cachefix_v2' not in gqa_provenance.get('cache', ''):
    raise RuntimeError('GQA validation cache is not the corrected cachefix_v2 cache')
if 'mha_provenance' in globals() and 'cachefix_v2' not in mha_provenance.get('cache', ''):
    raise RuntimeError('MHA validation cache is not the corrected cachefix_v2 cache')

# Fail closed unless both complete caches contain every word's logits on CPU.
for _name, _cache in (('GQA', gqa_val), ('MHA', mha_val)):
    if not _cache.get('complete') or len(_cache['emissions']) != len(val_windows):
        raise ValueError(f'{_name} cache is incomplete')
    for _i, (_window, _logits, _path) in enumerate(
            zip(val_windows, _cache['emissions'], _cache['reference_paths'])):
        _expected = (len(_window['word_last']), len(TAGS))
        if tuple(_logits.shape) != _expected or len(_path) != _expected[0]:
            raise ValueError(f'{_name} cache shape/path mismatch at window {_i}: '
                             f'{tuple(_logits.shape)} / {len(_path)} vs {_expected}')
        if _logits.device.type != 'cpu':
            raise ValueError('Pilot must use cached CPU tensors only')

# Fixed split, selected once by sorted document ID; no alternate split search.
_doc_ids = sorted(str(row['id']) for row in val_rows)
if len(_doc_ids) != 6 or len(set(_doc_ids)) != 6:
    raise ValueError(f'Expected the original six distinct validation docs, got {_doc_ids}')
_META_IDS, _HOLDOUT_IDS = set(_doc_ids[:3]), set(_doc_ids[3:])
_META_WINDOWS = [w for w in val_windows if str(w['document_id']) in _META_IDS]
_HOLDOUT_WINDOWS = [w for w in val_windows if str(w['document_id']) in _HOLDOUT_IDS]
_META_ROWS = [r for r in val_rows if str(r['id']) in _META_IDS]
_HOLDOUT_ROWS = [r for r in val_rows if str(r['id']) in _HOLDOUT_IDS]
if not _META_WINDOWS or not _HOLDOUT_WINDOWS:
    raise ValueError('Fixed 3/3 document split produced an empty side')

# One differentiable scalar CRF NLL over complete blended scores. Gold segments
# come from the project's existing BIO ambiguity/alignment routine.
_gcomp, _mcomp = gqa_val['components'], mha_val['components']
if not torch.equal(_gcomp['allowed'], _mcomp['allowed']) or not torch.equal(
        _gcomp['start_allowed'], _mcomp['start_allowed']):
    raise ValueError('GQA/MHA BIO legality masks differ')
def _normal_cpu_tensor(_tensor):
    # Cached inference tensors cannot be saved by autograd for backward. Clone
    # with inference mode explicitly disabled; this is CPU-only and done once.
    with torch.inference_mode(False):
        return _tensor.detach().to('cpu').clone()

_g_trans, _m_trans = _normal_cpu_tensor(_gcomp['transitions']), _normal_cpu_tensor(_mcomp['transitions'])
_g_start, _m_start = _normal_cpu_tensor(_gcomp['start']), _normal_cpu_tensor(_mcomp['start'])
_g_end, _m_end = _normal_cpu_tensor(_gcomp['end']), _normal_cpu_tensor(_mcomp['end'])
_allowed = _normal_cpu_tensor(_gcomp['allowed']).bool()
_start_allowed = _normal_cpu_tensor(_gcomp['start_allowed']).bool()
_G_EMISSIONS = [_normal_cpu_tensor(x) for x in gqa_val['emissions']]
_M_EMISSIONS = [_normal_cpu_tensor(x) for x in mha_val['emissions']]

def _meta_objective(_alpha):
    _one = 1.0 - _alpha
    _trans = (_alpha * _g_trans + _one * _m_trans)
    _trans = _trans.masked_fill(~_allowed, -1e4)
    _start = (_alpha * _g_start + _one * _m_start)
    _start = _start.masked_fill(~_start_allowed, -1e4)
    _end = _alpha * _g_end + _one * _m_end
    _window_losses = []
    for _wi, _window in enumerate(val_windows):
        if str(_window['document_id']) not in _META_IDS:
            continue
        _ge, _me = _G_EMISSIONS[_wi], _M_EMISSIONS[_wi]
        _em = _alpha * _ge + _one * _me
        _targets, _segments, _ambiguous = targets_and_segments(_window, LABELS)
        _segment_losses = []
        for _lo, _hi in _segments:
            if _hi <= _lo:
                continue
            _gold = _targets[_lo:_hi].long()
            _score = _start + _em[_lo]
            for _t in range(1, _hi - _lo):
                _score = torch.logsumexp(_score[:, None] + _trans, dim=0) + _em[_lo + _t]
            _log_z = torch.logsumexp(_score + _end, dim=0)
            _gold_score = _start[_gold[0]] + _em[_lo, _gold[0]]
            for _t in range(1, len(_gold)):
                _gold_score = _gold_score + _trans[_gold[_t - 1], _gold[_t]] + _em[_lo + _t, _gold[_t]]
            _gold_score = _gold_score + _end[_gold[-1]]
            _segment_losses.append(_log_z - _gold_score)
        if _segment_losses:
            # Match the existing training policy: mean supervised segments per window.
            _window_losses.append(torch.stack(_segment_losses).mean())
    if not _window_losses:
        raise ValueError('No supervised BIO segments in fixed meta-fit documents')
    _nll = torch.stack(_window_losses).mean()
    # Fixed shrinkage toward neutral 50/50; lambda is declared before fitting.
    return _nll + 0.1 * (_alpha - 0.5).square(), _nll

# Convex 1-D objective: derivative bisection, 16 iterations maximum; no grid search.
with torch.enable_grad():
    _left, _right = 0.0, 1.0
    _mid = torch.tensor(0.5, dtype=torch.float64, requires_grad=True)
    _obj, _ = _meta_objective(_mid)
    _grad, = torch.autograd.grad(_obj, _mid)
    if _grad.item() >= 0:
        _learned_weight = 0.0
    else:
        _at_one = torch.tensor(1.0, dtype=torch.float64, requires_grad=True)
        _obj1, _ = _meta_objective(_at_one)
        _grad1, = torch.autograd.grad(_obj1, _at_one)
        if _grad1.item() <= 0:
            _learned_weight = 1.0
        else:
            for _ in range(16):
                _mid_value = (_left + _right) / 2
                _probe = torch.tensor(_mid_value, dtype=torch.float64, requires_grad=True)
                _objective, _ = _meta_objective(_probe)
                _derivative, = torch.autograd.grad(_objective, _probe)
                if _derivative.item() < 0:
                    _left = _mid_value
                else:
                    _right = _mid_value
            _learned_weight = (_left + _right) / 2

# Evaluate only the predeclared holdout documents at learned alpha, 0.5, and 1.0.
def _subset_cache(_cache, _wanted):
    _indices = [i for i, w in enumerate(val_windows)
                if str(w['document_id']) in _wanted]
    return {**_cache,
            'emissions': [_cache['emissions'][i] for i in _indices],
            'reference_paths': [_cache['reference_paths'][i] for i in _indices]}

_g_hold = _subset_cache(gqa_val, _HOLDOUT_IDS)
_m_hold = _subset_cache(mha_val, _HOLDOUT_IDS)
_pilot_scores = {}
for _weight in (float(_learned_weight), 0.5, 1.0):
    _key = f'{_weight:.6f}'
    _pilot_scores[_key] = evaluate_weight(
        _HOLDOUT_WINDOWS, _HOLDOUT_ROWS, _g_hold, _m_hold, _weight,
        LABELS, TAGS, decode_entities)

with torch.no_grad():
    _fit_objective, _fit_nll = _meta_objective(torch.tensor(
        float(_learned_weight), dtype=torch.float64))
print('CPU-only scalar-mixture pilot; no checkpoints, runtime settings, or export paths changed.')
print('Fixed meta-fit document IDs:', sorted(_META_IDS), 'windows:', len(_META_WINDOWS))
print('Fixed holdout document IDs:', sorted(_HOLDOUT_IDS), 'windows:', len(_HOLDOUT_WINDOWS))
print('Regularized CRF NLL fit: GQA weight=', float(_learned_weight),
      'MHA weight=', 1.0 - float(_learned_weight), 'meta NLL=', float(_fit_nll),
      'objective=', float(_fit_objective), 'shrinkage_lambda=0.1', 'iterations<=16')
for _weight_key, _result in _pilot_scores.items():
    print('holdout GQA weight', _weight_key, 'macro-F1=', _result['macro_f1'],
          'per-label=', _result['per_label'])
print('CONTAMINATION: this is exploratory only. All six original validation documents '
      'were previously used for checkpoint/mixture review; the three-doc holdout is not an '
      'independent estimate. Do not use this result to replace the already selected '
      '50/50 Phase-2 test export or claim unbiased generalization.')


CPU-only scalar-mixture pilot; no checkpoints, runtime settings, or export paths changed.
Fixed meta-fit document IDs: ['13618343', '1389243', '1622258'] windows: 61
Fixed holdout document IDs: ['1680408', '50547742', '736324'] windows: 50
Regularized CRF NLL fit: GQA weight= 0.0 MHA weight= 1.0 meta NLL= 18.852741241455078 objective= 18.877741241455077 shrinkage_lambda=0.1 iterations<=16
holdout GQA weight 0.000000 macro-F1= 0.37691985604193756 per-label= {'APP': {'gold': 15, 'predicted': 6, 'tp': 0, 'precision': 0.0, 'recall': 0.0, 'f1': 0.0}, 'RESP': {'gold': 5, 'predicted': 12, 'tp': 3, 'precision': 0.25, 'recall': 0.6, 'f1': 0.35294117647058826}, 'A.COUNSEL': {'gold': 11, 'predicted': 7, 'tp': 3, 'precision': 0.42857142857142855, 'recall': 0.2727272727272727, 'f1': 0.33333333333333326}, 'R.COUNSEL': {'gold': 8, 'predicted': 1, 'tp': 0, 'precision': 0.0, 'recall': 0.0, 'f1': 0.0}, 'JUDGE': {'gold': 7, 'predicted': 12, 'tp': 2, 'precision': 0.16666666666666666, 'recall': 0.285714285

In [13]:
# Tiny CPU-only pilot for the already recovered six-document validation caches.
# Paste/run this as one new cell in the recovered ensemble Colab runtime.
# It does not load models, run GPU work, change ACTION/RUN_ROOT, write caches,
# or touch the in-progress 50/50 test export.
import torch

_REQUIRED = ('gqa_val', 'mha_val', 'val_windows', 'val_rows', 'LABELS', 'TAGS',
             'targets_and_segments', 'decode_entities', 'evaluate_weight')
_missing = [name for name in _REQUIRED if name not in globals()]
if _missing:
    raise RuntimeError(f'Open the recovered ensemble runtime first; missing: {_missing}')
if 'gqa_provenance' in globals() and 'cachefix_v2' not in gqa_provenance.get('cache', ''):
    raise RuntimeError('GQA validation cache is not the corrected cachefix_v2 cache')
if 'mha_provenance' in globals() and 'cachefix_v2' not in mha_provenance.get('cache', ''):
    raise RuntimeError('MHA validation cache is not the corrected cachefix_v2 cache')

# Fail closed unless both complete caches contain every word's logits on CPU.
for _name, _cache in (('GQA', gqa_val), ('MHA', mha_val)):
    if not _cache.get('complete') or len(_cache['emissions']) != len(val_windows):
        raise ValueError(f'{_name} cache is incomplete')
    for _i, (_window, _logits, _path) in enumerate(
            zip(val_windows, _cache['emissions'], _cache['reference_paths'])):
        _expected = (len(_window['word_last']), len(TAGS))
        if tuple(_logits.shape) != _expected or len(_path) != _expected[0]:
            raise ValueError(f'{_name} cache shape/path mismatch at window {_i}: '
                             f'{tuple(_logits.shape)} / {len(_path)} vs {_expected}')
        if _logits.device.type != 'cpu':
            raise ValueError('Pilot must use cached CPU tensors only')

# Fixed split, selected once by sorted document ID; no alternate split search.
_doc_ids = sorted(str(row['id']) for row in val_rows)
if len(_doc_ids) != 6 or len(set(_doc_ids)) != 6:
    raise ValueError(f'Expected the original six distinct validation docs, got {_doc_ids}')
_META_IDS, _HOLDOUT_IDS = set(_doc_ids[:3]), set(_doc_ids[3:])
_META_WINDOWS = [w for w in val_windows if str(w['document_id']) in _META_IDS]
_HOLDOUT_WINDOWS = [w for w in val_windows if str(w['document_id']) in _HOLDOUT_IDS]
_META_ROWS = [r for r in val_rows if str(r['id']) in _META_IDS]
_HOLDOUT_ROWS = [r for r in val_rows if str(r['id']) in _HOLDOUT_IDS]
if not _META_WINDOWS or not _HOLDOUT_WINDOWS:
    raise ValueError('Fixed 3/3 document split produced an empty side')

# One differentiable scalar CRF NLL over complete blended scores. Gold segments
# come from the project's existing BIO ambiguity/alignment routine.
_gcomp, _mcomp = gqa_val['components'], mha_val['components']
if not torch.equal(_gcomp['allowed'], _mcomp['allowed']) or not torch.equal(
        _gcomp['start_allowed'], _mcomp['start_allowed']):
    raise ValueError('GQA/MHA BIO legality masks differ')
def _normal_cpu_tensor(_tensor):
    # Cached inference tensors cannot be saved by autograd for backward. Clone
    # with inference mode explicitly disabled; this is CPU-only and done once.
    with torch.inference_mode(False):
        return _tensor.detach().to('cpu').clone()

_g_trans, _m_trans = _normal_cpu_tensor(_gcomp['transitions']), _normal_cpu_tensor(_mcomp['transitions'])
_g_start, _m_start = _normal_cpu_tensor(_gcomp['start']), _normal_cpu_tensor(_mcomp['start'])
_g_end, _m_end = _normal_cpu_tensor(_gcomp['end']), _normal_cpu_tensor(_mcomp['end'])
_allowed = _normal_cpu_tensor(_gcomp['allowed']).bool()
_start_allowed = _normal_cpu_tensor(_gcomp['start_allowed']).bool()
_G_EMISSIONS = [_normal_cpu_tensor(x) for x in gqa_val['emissions']]
_M_EMISSIONS = [_normal_cpu_tensor(x) for x in mha_val['emissions']]

def _meta_objective(_alpha):
    _one = 1.0 - _alpha
    _trans = (_alpha * _g_trans + _one * _m_trans)
    _trans = _trans.masked_fill(~_allowed, -1e4)
    _start = (_alpha * _g_start + _one * _m_start)
    _start = _start.masked_fill(~_start_allowed, -1e4)
    _end = _alpha * _g_end + _one * _m_end
    _window_losses = []
    for _wi, _window in enumerate(val_windows):
        if str(_window['document_id']) not in _META_IDS:
            continue
        _ge, _me = _G_EMISSIONS[_wi], _M_EMISSIONS[_wi]
        _em = _alpha * _ge + _one * _me
        _targets, _segments, _ambiguous = targets_and_segments(_window, LABELS)
        _segment_losses = []
        for _lo, _hi in _segments:
            if _hi <= _lo:
                continue
            _gold = _targets[_lo:_hi].long()
            _score = _start + _em[_lo]
            for _t in range(1, _hi - _lo):
                _score = torch.logsumexp(_score[:, None] + _trans, dim=0) + _em[_lo + _t]
            _log_z = torch.logsumexp(_score + _end, dim=0)
            _gold_score = _start[_gold[0]] + _em[_lo, _gold[0]]
            for _t in range(1, len(_gold)):
                _gold_score = _gold_score + _trans[_gold[_t - 1], _gold[_t]] + _em[_lo + _t, _gold[_t]]
            _gold_score = _gold_score + _end[_gold[-1]]
            _segment_losses.append(_log_z - _gold_score)
        if _segment_losses:
            # Match the existing training policy: mean supervised segments per window.
            _window_losses.append(torch.stack(_segment_losses).mean())
    if not _window_losses:
        raise ValueError('No supervised BIO segments in fixed meta-fit documents')
    _nll = torch.stack(_window_losses).mean()
    # Fixed shrinkage toward neutral 50/50; lambda is declared before fitting.
    return _nll + 0.1 * (_alpha - 0.5).square(), _nll

# Convex 1-D objective: first check endpoint derivatives, then bisect the
# interior root. The derivative at 0.5 alone cannot determine the minimizer.
def _bounded_convex_minimize(_objective_fn, _steps=16):
    def _derivative(_value):
        with torch.enable_grad():
            _x = torch.tensor(float(_value), dtype=torch.float64, requires_grad=True)
            _y = _objective_fn(_x)
            return float(torch.autograd.grad(_y, _x)[0].item())
    _g0, _g1 = _derivative(0.0), _derivative(1.0)
    if _g0 >= 0.0:
        return 0.0
    if _g1 <= 0.0:
        return 1.0
    _left, _right = 0.0, 1.0
    for _ in range(_steps):
        _mid = (_left + _right) / 2
        if _derivative(_mid) < 0.0:
            _left = _mid
        else:
            _right = _mid
    return (_left + _right) / 2

# Cheap optimizer regression checks: two interior optima and both boundaries.
for _target, _expected in ((0.2, 0.2), (0.8, 0.8), (-0.3, 0.0), (1.3, 1.0)):
    _got = _bounded_convex_minimize(lambda _x, t=_target: (_x - t).square(), 20)
    assert abs(_got - _expected) < 2e-6, f'bounded scalar optimizer regression: {_target} -> {_got}'

_learned_weight = _bounded_convex_minimize(lambda _a: _meta_objective(_a)[0], 16)

# Evaluate only the predeclared holdout documents at learned alpha, 0.5, and 1.0.
def _subset_cache(_cache, _wanted):
    _indices = [i for i, w in enumerate(val_windows)
                if str(w['document_id']) in _wanted]
    return {**_cache,
            'emissions': [_cache['emissions'][i] for i in _indices],
            'reference_paths': [_cache['reference_paths'][i] for i in _indices]}

_g_hold = _subset_cache(gqa_val, _HOLDOUT_IDS)
_m_hold = _subset_cache(mha_val, _HOLDOUT_IDS)
_pilot_scores = {}
for _weight in (float(_learned_weight), 0.5, 1.0):
    _key = f'{_weight:.6f}'
    _pilot_scores[_key] = evaluate_weight(
        _HOLDOUT_WINDOWS, _HOLDOUT_ROWS, _g_hold, _m_hold, _weight,
        LABELS, TAGS, decode_entities)

with torch.no_grad():
    _fit_objective, _fit_nll = _meta_objective(torch.tensor(
        float(_learned_weight), dtype=torch.float64))
print('CPU-only scalar-mixture pilot; no checkpoints, runtime settings, or export paths changed.')
print('Fixed meta-fit document IDs:', sorted(_META_IDS), 'windows:', len(_META_WINDOWS))
print('Fixed holdout document IDs:', sorted(_HOLDOUT_IDS), 'windows:', len(_HOLDOUT_WINDOWS))
print('Regularized CRF NLL fit: GQA weight=', float(_learned_weight),
      'MHA weight=', 1.0 - float(_learned_weight), 'meta NLL=', float(_fit_nll),
      'objective=', float(_fit_objective), 'shrinkage_lambda=0.1', 'iterations<=16')
for _weight_key, _result in _pilot_scores.items():
    print('holdout GQA weight', _weight_key, 'macro-F1=', _result['macro_f1'],
          'per-label=', _result['per_label'])
print('CONTAMINATION: this is exploratory only. All six original validation documents '
      'were previously used for checkpoint/mixture review; the three-doc holdout is not an '
      'independent estimate. Do not use this result to replace the already selected '
      '50/50 Phase-2 test export or claim unbiased generalization.')


CPU-only scalar-mixture pilot; no checkpoints, runtime settings, or export paths changed.
Fixed meta-fit document IDs: ['13618343', '1389243', '1622258'] windows: 61
Fixed holdout document IDs: ['1680408', '50547742', '736324'] windows: 50
Regularized CRF NLL fit: GQA weight= 0.0 MHA weight= 1.0 meta NLL= 18.852741241455078 objective= 18.877741241455077 shrinkage_lambda=0.1 iterations<=16
holdout GQA weight 0.000000 macro-F1= 0.37691985604193756 per-label= {'APP': {'gold': 15, 'predicted': 6, 'tp': 0, 'precision': 0.0, 'recall': 0.0, 'f1': 0.0}, 'RESP': {'gold': 5, 'predicted': 12, 'tp': 3, 'precision': 0.25, 'recall': 0.6, 'f1': 0.35294117647058826}, 'A.COUNSEL': {'gold': 11, 'predicted': 7, 'tp': 3, 'precision': 0.42857142857142855, 'recall': 0.2727272727272727, 'f1': 0.33333333333333326}, 'R.COUNSEL': {'gold': 8, 'predicted': 1, 'tp': 0, 'precision': 0.0, 'recall': 0.0, 'f1': 0.0}, 'JUDGE': {'gold': 7, 'predicted': 12, 'tp': 2, 'precision': 0.16666666666666666, 'recall': 0.285714285